In [0]:
%run "/Workspace/ETL_ARQUITETURA_MEDALHAO/00.config/config"

In [0]:
from pyspark.sql import functions as F, Window

df = spark.table(f"{CATALOG}.{SILVER}.economia")


df.display()


# COMMAND ----------

from pyspark.sql import functions as F, Window

df = spark.table(f"{CATALOG}.{SILVER}.economia")

w = Window.orderBy("data")

data,ipca,boi_gordo,data_coleta
2024-01-01,0.42,249.65,2026-05-07T01:58:45.427Z
2024-02-01,0.83,237.84,2026-05-07T01:58:45.427Z
2024-03-01,0.16,232.81,2026-05-07T01:58:45.427Z
2024-04-01,0.38,230.51,2026-05-07T01:58:45.427Z
2024-05-01,0.46,226.92,2026-05-07T01:58:45.427Z
2024-06-01,0.21,220.7,2026-05-07T01:58:45.427Z
2024-07-01,0.38,229.27,2026-05-07T01:58:45.427Z
2024-08-01,-0.02,235.07,2026-05-07T01:58:45.427Z
2024-09-01,0.44,255.45,2026-05-07T01:58:45.427Z
2024-10-01,0.56,301.13,2026-05-07T01:58:45.427Z


In [0]:
gold = (df
    .withColumn("ipca_ant", F.lag("ipca").over(w))
    .withColumn("boi_ant", F.lag("boi_gordo").over(w))
    .withColumn("variacao_ipca", (F.col("ipca") - F.col("ipca_ant")) / F.col("ipca_ant") * 100)
    .withColumn("variacao_boi", (F.col("boi_gordo") - F.col("boi_ant")) / F.col("boi_ant") * 100)
    .drop("ipca_ant", "boi_ant")
)

gold.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD}.insights")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
SELECT * FROM workspace.gold_economia.insights LIMIT 20;

data,ipca,boi_gordo,data_coleta,variacao_ipca,variacao_boi
2024-01-01,0.42,249.65,2026-05-07T01:58:45.427Z,null,null
2024-02-01,0.83,237.84,2026-05-07T01:58:45.427Z,97.61904761904762,-4.73062287202083
2024-03-01,0.16,232.81,2026-05-07T01:58:45.427Z,-80.72289156626505,-2.1148671375714767
2024-04-01,0.38,230.51,2026-05-07T01:58:45.427Z,137.5,-0.9879300717323187
2024-05-01,0.46,226.92,2026-05-07T01:58:45.427Z,21.052631578947373,-1.5574161641577389
2024-06-01,0.21,220.7,2026-05-07T01:58:45.427Z,-54.347826086956516,-2.741054115988013
2024-07-01,0.38,229.27,2026-05-07T01:58:45.427Z,80.95238095238096,3.8830992297236167
2024-08-01,-0.02,235.07,2026-05-07T01:58:45.427Z,-105.26315789473684,2.529768395341729
2024-09-01,0.44,255.45,2026-05-07T01:58:45.427Z,-2300.0,8.669757944442079
2024-10-01,0.56,301.13,2026-05-07T01:58:45.427Z,27.27272727272728,17.88216872186338


Catálogo: workspace
Schemas: bronze_economia silver_economia gold_economia
